# Avery Q8 → Q4_K_M Quantization
Converts `avery-sovereign-q8.gguf` (8.1GB) to `avery-sovereign-q4km.gguf` (~4.5GB)
so it fits in 8GB VRAM on RTX 5050.

**Before running:** Add HF_TOKEN to Kaggle Secrets (Settings → Add-ons → Secrets)

In [ ]:
# Cell 1 — Setup
import os, subprocess, shutil

# Load HF token from Kaggle secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
print('HF token loaded ✅')

# Check disk space
total, used, free = shutil.disk_usage('/')
print(f'Disk: {free/1e9:.1f} GB free of {total/1e9:.1f} GB total')

In [ ]:
# Cell 2 — Install dependencies
subprocess.run(['pip', 'install', '-q', 'huggingface_hub'], check=True)
print('huggingface_hub installed ✅')

In [ ]:
# Cell 3 — Download llama.cpp Linux binary (has llama-quantize)
import urllib.request, zipfile

LLAMA_URL = 'https://github.com/ggml-org/llama.cpp/releases/download/b9371/llama-b9371-bin-ubuntu-x64.zip'
LLAMA_ZIP = '/kaggle/working/llama-cpp-linux.zip'
LLAMA_DIR = '/kaggle/working/llama-cpp/'

print('Downloading llama.cpp Linux binary...')
urllib.request.urlretrieve(LLAMA_URL, LLAMA_ZIP)
print(f'Downloaded: {os.path.getsize(LLAMA_ZIP)/1e6:.1f} MB')

os.makedirs(LLAMA_DIR, exist_ok=True)
with zipfile.ZipFile(LLAMA_ZIP, 'r') as z:
    z.extractall(LLAMA_DIR)

quantize_exe = os.path.join(LLAMA_DIR, 'llama-quantize')
os.chmod(quantize_exe, 0o755)
print(f'llama-quantize ready: {quantize_exe} ✅')

In [ ]:
# Cell 4 — Download Q8 GGUF from HuggingFace
from huggingface_hub import hf_hub_download

Q8_PATH = '/kaggle/working/avery-sovereign-q8.gguf'
Q4KM_PATH = '/kaggle/working/avery-sovereign-q4km.gguf'

print('Downloading avery-sovereign-q8.gguf from HuggingFace...')
print('This is 8.1GB — takes ~5-10 minutes on Kaggle')

downloaded = hf_hub_download(
    repo_id='tastytator/avery-sovereign-lora',
    filename='avery-sovereign-q8.gguf',
    local_dir='/kaggle/working/',
    token=HF_TOKEN
)
print(f'Downloaded to: {downloaded}')
print(f'Size: {os.path.getsize(Q8_PATH)/1e9:.2f} GB ✅')

In [ ]:
# Cell 5 — Quantize Q8 → Q4_K_M
import time

print('Starting quantization: Q8_0 → Q4_K_M')
print('Expected output: ~4.5 GB')
print('Expected time: 5-15 minutes...')

t0 = time.time()
result = subprocess.run(
    [quantize_exe, Q8_PATH, Q4KM_PATH, 'Q4_K_M'],
    capture_output=True,
    text=True
)
elapsed = time.time() - t0

if result.returncode == 0:
    size = os.path.getsize(Q4KM_PATH)
    print(f'Quantization complete in {elapsed:.1f}s ✅')
    print(f'Output: {Q4KM_PATH}')
    print(f'Size: {size/1e9:.2f} GB')
else:
    print('ERROR:')
    print(result.stdout[-2000:])
    print(result.stderr[-2000:])

In [ ]:
# Cell 6 — Upload Q4KM to HuggingFace
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
REPO_ID = 'tastytator/avery-sovereign-lora'

print(f'Uploading avery-sovereign-q4km.gguf to {REPO_ID}...')
print('This is ~4.5GB — takes ~5-10 minutes...')

t0 = time.time()
api.upload_file(
    path_or_fileobj=Q4KM_PATH,
    path_in_repo='avery-sovereign-q4km.gguf',
    repo_id=REPO_ID,
    repo_type='model',
    commit_message='Add Q4_K_M quantized GGUF for 8GB VRAM deployment'
)
elapsed = time.time() - t0
print(f'Upload complete in {elapsed:.1f}s ✅')
print(f'Available at: https://huggingface.co/{REPO_ID}/blob/main/avery-sovereign-q4km.gguf')

In [ ]:
# Cell 7 — Verify upload
from huggingface_hub import list_repo_files

print(f'Files in {REPO_ID}:')
for f in list_repo_files(REPO_ID, token=HF_TOKEN):
    print(f'  {f}')

print()
print('=== ALL DONE ===')
print('Next steps on your local machine:')
print('1. Delete C:/Users/leer4/GH05T3/avery-sovereign-q8.gguf  (frees 8.1 GB)')
print('2. Run: python -c "from huggingface_hub import hf_hub_download; hf_hub_download(repo_id=\'tastytator/avery-sovereign-lora\', filename=\'avery-sovereign-q4km.gguf\', local_dir=\'C:/Users/leer4/GH05T3/\')')
print('3. Edit Modelfile.avery: change FROM to avery-sovereign-q4km.gguf')
print('4. Run: ollama create avery-sovereign -f C:/Users/leer4/GH05T3/Modelfile.avery')
print('5. Test: ollama run avery-sovereign "Give me a KAIROS kickoff for a CPA firm"')